In [0]:
%python
import pandas as pd

# --- Read source CSV from landing zone ---
src_path = "/Volumes/workspace/default/ftw-b12-de/groups/g5_landing/facilities/authorized_bed_capacity_2020_2024_asof_20260929.csv"
df = pd.read_csv(src_path, sep=";", header=1)

# --- Unpivot wide format to long format ---
# Source columns: Regions, Total 2020..2024, Government 2020..2024, Private 2020..2024
melted = df.melt(
    id_vars=["Regions"],
    var_name="col_name",
    value_name="authorized_bed_capacity",
)

# Extract year (e.g. "Total 2020" -> 2020)
melted["year"] = melted["col_name"].str.extract(r"(\d{4})").astype(int)

# Extract nature of ownership (e.g. "Government 2020" -> "government", "Total 2020" -> "total")
melted["nature_of_ownership"] = (
    melted["col_name"]
    .str.replace(r"\s*\d{4}$", "", regex=True)
    .str.lower()
)

# Rename and select final columns
melted = melted.rename(columns={"Regions": "region"})
melted = melted[["region", "year", "nature_of_ownership", "authorized_bed_capacity"]]
melted = melted.sort_values(["region", "year", "nature_of_ownership"]).reset_index(drop=True)

# Convert object columns to nullable string for Arrow compatibility
melted["region"] = melted["region"].astype("string")
melted["nature_of_ownership"] = melted["nature_of_ownership"].astype("string")

# --- Write to Delta table ---
table_name = "ph_public_health.bronze.facilities_authorized_bed_capacities"
spark_df = spark.createDataFrame(melted)
(
    spark_df.write.format("delta")
    .mode("overwrite")
    .option("columnMapping.mode", "name")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
)

print(f"Wrote {melted.shape[0]} rows to {table_name}")
display(spark.table(table_name).limit(20))